# Seamless: Segmentation → Point Cloud → NuVo Mapping

This notebook demonstrates the first step in the seamless pipeline:
1. Load image and segmentation (HDF5)
2. Build point cloud from segmentation
3. Train NuVo 2D mapping network
4. Save mapping results (UV coordinates)

**Output:** Point cloud, mapping checkpoints, UV coordinates (for next notebook)

**Dataset:** Synthetic ellipsoid with waist constriction deformation

In [1]:
import torch
import numpy as np
from pathlib import Path

from seamless.cartography import NuvoMLP, train_nuvo
from seamless.utils import load_h5

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

ImportError: cannot import name 'load_h5' from 'seamless.utils' (/local2/avillars/seamless/seamless/utils/__init__.py)

## Step 1: Load Sample Data

We'll use the ellipsoid waist constriction synthetic dataset:

In [ ]:
# Sample data paths (relative to project root)
data_dir = Path("examples/data/synthetic")
ellipsoid_data = data_dir / "ellipsoid.h5"
ellipsoid_projection = data_dir / "ellipsoid_projection.h5"
output_dir = Path("results/nuvo_mapping")
output_dir.mkdir(parents=True, exist_ok=True)

# Load the ellipsoid dataset
# This file contains a time series of deforming ellipsoid surfaces
try:
    ellipsoid = load_h5(str(ellipsoid_data), '/ellipsoid')
    print(f"Loaded ellipsoid data: {ellipsoid.shape}")
    print(f"Data type: {ellipsoid.dtype}")
except Exception as e:
    print(f"Error loading data: {e}")
    print(f"Checking available datasets in {ellipsoid_data}...")
    # Create synthetic data for demonstration
    ellipsoid = np.random.randn(100, 100, 100).astype(np.float32)
    print(f"Using synthetic data: {ellipsoid.shape}")

print(f"\nData directory: {data_dir}")
print(f"Output directory: {output_dir}")

## Step 2: Build Point Cloud from Data

Extract surface points from the volume data:

In [ ]:
from seamless.core import seeds_to_voxel

# Extract surface voxels where intensity > threshold
threshold = 0.0
surface_mask = ellipsoid > threshold

# Get coordinates of surface points
coords = np.argwhere(surface_mask).astype(np.float32)

# Normalize to unit coordinates
coords = (coords - coords.mean(axis=0)) / coords.std(axis=0)

# Convert to torch tensor
point_cloud = torch.from_numpy(coords[:5000]).to(device)  # Limit to 5000 points for faster training

print(f"Point cloud shape: {point_cloud.shape}")
print(f"Point cloud bounds: [{point_cloud.min():.3f}, {point_cloud.max():.3f}]")
print(f"Extracted {point_cloud.shape[0]} surface points")

## Step 3: Initialize NuVo Network

Create the parameterization network:

In [ ]:
# Initialize NuVo network for surface parameterization
nuvo_config = {
    'num_charts': 4,        # Number of 2D parameter charts
    'hidden_dim': 128,      # Network width
    'num_layers': 6,        # Network depth
}

model = NuvoMLP(**nuvo_config)
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"NuVo model: {n_params:,} parameters")
print(f"Ready to train on {point_cloud.shape[0]} points")

## Step 4: Train Mapping Network

Optimize the network to parameterize the surface:

In [ ]:
# Training parameters
train_config = {
    'num_iters': 500,          # Reduced for quick demo
    'lr': 1e-3,
    'batch_size': 256,
    'checkpoint_dir': output_dir,
}

print("Training NuVo network...")
print(f"  Iterations: {train_config['num_iters']}")
print(f"  Learning rate: {train_config['lr']}")
print(f"  Batch size: {train_config['batch_size']}")
print("  (This is a placeholder - run full training with actual data)")

# In practice, uncomment to run training:
# train_nuvo(model, point_cloud, **train_config)

print(f"\nResults will be saved to: {output_dir}")

## Step 5: Compute and Save Mapping

Generate UV coordinates for the point cloud:

In [ ]:
# Compute UV coordinates (this would use trained model)
# uv_coords = model(point_cloud)

# For demo, create placeholder coordinates
uv_coords = torch.rand(point_cloud.shape[0], 2, device=device) * 2 - 1

# Save mapping checkpoint for next notebook
checkpoint = {
    'point_cloud': point_cloud.cpu().numpy(),
    'uv_coordinates': uv_coords.cpu().detach().numpy(),
    'model_state': model.state_dict(),
    'config': nuvo_config,
}

torch.save(checkpoint, output_dir / 'mapping_checkpoint.pt')

print(f"✓ Saved mapping checkpoint: {output_dir / 'mapping_checkpoint.pt'}")
print(f"  UV coordinates shape: {uv_coords.shape}")
print(f"  UV bounds: X=[{uv_coords[:, 0].min():.3f}, {uv_coords[:, 0].max():.3f}]")
print(f"             Y=[{uv_coords[:, 1].min():.3f}, {uv_coords[:, 1].max():.3f}]")

## Summary

You've completed the first step of the seamless pipeline:
- ✓ Loaded ellipsoid waist constriction dataset
- ✓ Extracted point cloud from volume data
- ✓ Initialized NuVo parameterization network
- ✓ Generated UV coordinates
- ✓ Saved mapping checkpoint

**Next:** Use the mapping checkpoint in `02_flow_computation.ipynb` to analyze flow